# Lábia — teste de fumaça do treino (SFT com QLoRA no aluno)

Antes do treino de verdade, confirmamos que o treino **funciona nesta máquina**, com o formato exato dos nossos dados. Três perguntas:

1. O aluno (**Qwen3-4B-Instruct-2507**, Apache 2.0) carrega comprimido em 4 bits numa T4 e treina sem estourar a memória?
2. Quantos tokens por segundo o treino processa? Esse número decide quantas conversas cabem nas horas de GPU.
3. O modelo-base, sem ajuste nenhum, já sabe chamar as nossas ferramentas em português? É a linha de base do projeto.

Conceitos usados aqui:

- **SFT (ajuste fino supervisionado):** mostrar ao modelo exemplos de "conversa → resposta certa" e ajustá-lo para imitar.
- **LoRA:** em vez de mexer nos 4 bilhões de parâmetros, treinamos pequenas matrizes "acopladas" a eles (alguns milhões de parâmetros). É como aprender a tocar uma música nova sem reaprender o instrumento.
- **QLoRA:** LoRA com o modelo original guardado em 4 bits, o que corta a memória a um quarto.
- **float16:** a T4 não tem bfloat16 (achado do teste de fumaça do ambiente), então o treino usa float16.

Os dados deste teste vêm do **agente roteirizado "ideal"**, não do professor: o objetivo é testar o formato e a máquina, não ensinar nada. Por isso nenhum número de qualidade sai daqui.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # uma T4 só: o teste mede o caso simples, sem paralelismo
os.environ["TOKENIZERS_PARALLELISM"] = "false"
LABIA_COMMIT = "cf41166ad8233f258fd95b7de37a24c8ec9556f9"
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
MAX_LEN = 6144
STEPS = 20
OUT_DIR = "/kaggle/working/teste-treino"

## 1. Bibliotecas

O Kaggle já traz `transformers`, `peft` e `accelerate`. Faltam a `trl` e a `bitsandbytes` (a biblioteca que comprime em 4 bits), como o teste de fumaça do ambiente mostrou.

In [ ]:
import subprocess
r = subprocess.run(["pip", "install", "-q", "bitsandbytes", "trl",
                    f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:])
import importlib.metadata as md_
for pkg in ["torch", "transformers", "peft", "bitsandbytes", "trl", "accelerate", "labia"]:
    try:
        print(f"{pkg:14} {md_.version(pkg)}")
    except md_.PackageNotFoundError:
        print(f"{pkg:14} AUSENTE")

## 2. Dados de formato real

Rodamos o agente roteirizado "ideal" em 64 cenários de treino e exportamos como exemplo de SFT, exatamente pela função que vai exportar as conversas do professor (`to_sft_example`, com a destilação de contexto). Se o formato estiver errado, quebramos aqui, e não no treino de verdade.

In [ ]:
import random
from labia.agents import ScriptedAgent, ScriptedCustomer
from labia.conversation import run_conversation
from labia.dataset import to_sft_example
from labia.scenarios.generator import Counts, generate

train = generate("train", Counts(train_per_cell=1))
random.Random(0).shuffle(train)
examples = []
for s in train[:64]:
    rec = run_conversation(s, ScriptedAgent(s, "ideal"), ScriptedCustomer(s), model_id="roteirizado")
    examples.append(to_sft_example(rec, s))
print(len(examples), "exemplos; primeiro:", examples[0]["id"])
print([m["role"] for m in examples[0]["messages"]])

## 3. Tokenizar com a máscara do agente

O modelo vê a conversa inteira, mas só **aprende as falas do agente** (texto e chamadas de ferramenta). A fala do cliente e o resultado das ferramentas são contexto. Sem essa máscara, o aluno também aprenderia a imitar o golpista.

A célula mostra o tamanho das conversas em tokens (decide o `MAX_LEN`) e um trecho do que é alvo de treino.

In [ ]:
import numpy as np
from transformers import AutoTokenizer
from labia.training import IGNORE, tokenize_for_sft

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
encoded, dropped = [], 0
for ex in examples:
    item = tokenize_for_sft(tokenizer, ex["messages"], ex["tools"], max_len=MAX_LEN)
    if item is None:
        dropped += 1
    else:
        encoded.append(item)
lengths = [len(e.input_ids) for e in encoded]
targets = [e.n_target for e in encoded]
print(f"{len(encoded)} exemplos, {dropped} descartados por passar de {MAX_LEN} tokens")
print(f"tokens por conversa: mediana {np.median(lengths):.0f}, p95 {np.percentile(lengths, 95):.0f}, máximo {max(lengths)}")
print(f"tokens de alvo (falas do agente): mediana {np.median(targets):.0f}, {sum(targets) / sum(lengths):.1%} do total")
sample = encoded[0]
target_ids = [t for t in sample.labels if t != IGNORE]
print("\n--- trecho do alvo de treino ---")
print(tokenizer.decode(target_ids)[:600])

## 4. Carregar o aluno em 4 bits e acoplar o LoRA

- `nf4`: o formato de 4 bits do QLoRA, pensado para pesos de redes neurais.
- `r=16`: o "tamanho" das matrizes do LoRA. Maior aprende mais, mas pesa mais.
- `target_modules`: o LoRA vai em todas as camadas lineares da atenção e do MLP.
- *gradient checkpointing*: recalcula partes da rede em vez de guardá-las, trocando tempo por memória.

In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

start = time.perf_counter()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, dtype=torch.float16,
                                             device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print(f"carregado em {time.perf_counter() - start:.0f} s; memória na GPU: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## 5. Antes de treinar: o modelo-base já chama as ferramentas?

Dez cenários de **teste**, só a primeira mensagem do cliente. Contamos quantas respostas trazem uma chamada de ferramenta que o nosso validador aceita. É um primeiro sinal da linha de base (o número oficial sai da avaliação completa).

In [ ]:
import json
from labia.agents import ScriptedCustomer
from labia.llm import extract_text_tool_calls
from labia.prompts import agent_system_prompt
from labia.sandbox import get_sector
from labia.sandbox.core import ToolCall, InvalidToolCall
from labia.contracts.common import Defense

test = generate("test", Counts(test_per_cell=1, test_heldout_per_cell=1))
random.Random(1).shuffle(test)
probe = test[:10]


def first_turn_calls(m, scenarios, max_new_tokens=200):
    m.eval()
    results = []
    for s in scenarios:
        sector = get_sector(s.sector)
        msgs = [{"role": "system", "content": agent_system_prompt(s, Defense.STANDARD)},
                {"role": "user", "content": ScriptedCustomer(s).next_message([])}]
        prompt = tokenizer.apply_chat_template(msgs, tools=sector.tool_schemas(), add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
        with torch.no_grad():
            out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
        _, calls = extract_text_tool_calls(text.replace("<|im_end|>", ""))
        valid = 0
        for c in calls:
            try:
                sector.validate(ToolCall(c.name, json.loads(c.arguments)))
                valid += 1
            except (InvalidToolCall, json.JSONDecodeError, TypeError):
                pass
        results.append({"scenario": s.id, "calls": [c.name for c in calls], "valid": valid, "text": text[:240]})
    return results

start = time.perf_counter()
before = first_turn_calls(model, probe)
print(f"{time.perf_counter() - start:.0f} s para 10 respostas")
for r in before:
    print(f"{r['valid']}/{len(r['calls'])} válidas  {r['calls']}  | {r['text'][:120]!r}")
print("respostas com pelo menos uma chamada válida:", sum(r["valid"] > 0 for r in before), "de", len(before))

## 6. Vinte passos de treino

Medimos o que importa para o planejamento: **tokens processados por segundo** e o **pico de memória**. A perda (*loss*) deve cair, mas com dados roteirizados e só 20 passos isso não diz nada sobre qualidade: só que o treino funciona.

In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments


class SFTData(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        e = self.items[i]
        return {"input_ids": e.input_ids, "attention_mask": e.attention_mask, "labels": e.labels}


def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    def fill(key, value):
        return torch.tensor([b[key] + [value] * (width - len(b[key])) for b in batch])
    return {"input_ids": fill("input_ids", pad), "attention_mask": fill("attention_mask", 0), "labels": fill("labels", IGNORE)}


args = TrainingArguments(
    output_dir=OUT_DIR, max_steps=STEPS, per_device_train_batch_size=1, gradient_accumulation_steps=4,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=2, logging_steps=5, fp16=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit", report_to="none", save_strategy="no", remove_unused_columns=False, seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=SFTData(encoded), data_collator=collate)
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
result = trainer.train()
seconds = time.perf_counter() - start
tokens_seen = sum(lengths[i % len(lengths)] for i in range(STEPS * 4))
stats = {
    "passos": STEPS, "segundos": round(seconds, 1),
    "tokens_por_segundo": round(tokens_seen / seconds, 1),
    "pico_memoria_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "loss_final": round(result.training_loss, 4),
    "log": [h for h in trainer.state.log_history if "loss" in h],
}
print(json.dumps(stats, indent=2))

## 7. Resumo

Guardamos tudo num arquivo. O planejamento do treino de verdade usa `tokens_por_segundo` para calcular quantas conversas cabem nas horas de GPU.

In [ ]:
from pathlib import Path
summary = {
    "modelo": BASE_MODEL, "max_len": MAX_LEN, "exemplos": len(encoded), "descartados_por_tamanho": dropped,
    "tokens_por_conversa_mediana": float(np.median(lengths)), "tokens_por_conversa_p95": float(np.percentile(lengths, 95)),
    "treino": stats,
    "base_primeiro_turno": {"com_chamada_valida": sum(r["valid"] > 0 for r in before), "de": len(before), "respostas": before},
    "dados": "agente roteirizado ideal (teste de formato; nenhuma conclusão de qualidade)",
}
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUT_DIR, "resumo.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in summary.items() if k != "base_primeiro_turno"}, ensure_ascii=False, indent=2))